# House Prices — подгруппа 1

**Задание:** `LabelEncoder` + `RandomForestRegressor`

Метрика: **RMSLE** (меньше = лучше).

Улучшения после разведки ошибок: признаки `Qual_x_SF`, `KitchenQualNum`, RF `n_estimators=500`.


In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


## 1. Данные


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# выбросы: огромная площадь + низкая цена
train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000))].reset_index(drop=True)

y = train["SalePrice"]
test_ids = test["Id"]
X = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])

print(X.shape, X_test.shape)


(1458, 79) (1459, 79)


## 2. Пропуски + признаки

Разведка ошибок на validation показала: модель часто промахивается на **старых дешёвых** домах и слабо учитывает связку «качество × размер» и качество кухни.

- `"None"` / `0` — где NA = нет объекта
- базовые: `TotalSF`, `TotalBath`, `Age`, `RemodAge`
- из разведки: `Qual_x_SF = OverallQual * TotalSF`, `KitchenQualNum` (Ex=5…Po=1)


In [3]:
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
for col in none_cols:
    X[col] = X[col].fillna("None")
    X_test[col] = X_test[col].fillna("None")

for col in ["GarageYrBlt", "MasVnrArea"]:
    X[col] = X[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

kitchen_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1}

for df in (X, X_test):
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"] = (
        df["FullBath"] + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"].fillna(0) + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["Qual_x_SF"] = df["OverallQual"] * df["TotalSF"]
    df["KitchenQualNum"] = df["KitchenQual"].map(kitchen_map).fillna(3)

num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median = X[col].median()
    X[col] = X[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("Missing").replace("nan", "Missing")
    X_test[col] = X_test[col].astype(str).fillna("Missing").replace("nan", "Missing")

print("пустых ячеек:", X.isna().sum().sum(), X_test.isna().sum().sum())


пустых ячеек: 0 0


## 3. LabelEncoder


In [4]:
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

X.head(3)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,MoSold,YrSold,SaleType,SaleCondition,TotalSF,TotalBath,Age,RemodAge,Qual_x_SF,KitchenQualNum
0,60,4,65.0,8450,1,1,3,3,0,4,...,2,2008,9,4,2566,3.5,5,5,17962,4
1,20,4,80.0,9600,1,1,3,3,0,2,...,5,2007,9,4,2524,2.5,31,31,15144,3
2,60,4,68.0,11250,1,1,0,3,0,4,...,9,2008,9,4,2706,3.5,7,6,18942,4


## 4. RandomForest + log-цена

Параметры подобраны перебором (`tune_rf.py`) на validation.


In [5]:
y_log = np.log1p(y)

X_tr, X_val, y_tr, y_val = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_val_price = np.expm1(y_val)

model = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.4,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_tr, y_tr)

pred = np.clip(np.expm1(model.predict(X_val)), 0, None)
rmsle = np.sqrt(mean_squared_log_error(y_val_price, pred))
print(f"Validation RMSLE: {rmsle:.5f}")


Validation RMSLE: 0.13606


## 5. Submission


In [6]:
final = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=1,
    min_samples_split=2,
    max_features=0.4,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
final.fit(X, y_log)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": np.clip(np.expm1(final.predict(X_test)), 0, None),
})
submission.to_csv("submission.csv", index=False)
submission.head()


,Id,SalePrice
0,1461,121900.184678
1,1462,157948.294762
2,1463,176681.835891
3,1464,183153.973106
4,1465,192254.281077
